# Vigour POC — Live Dashboard

**Top-down tracking + metric estimates** via 2D cone calibration (SAM3).

Run remotely via: `jupyter lab --ip 0.0.0.0 --port 8888 --no-browser --notebook-dir pipeline-poc/notebooks`

Then open `http://<server-ip>:8888` in your browser.

## 1. Run Pipeline

In [16]:
import sys, os, json, time
from pathlib import Path
from dataclasses import asdict


# ── Force GPU 0 (RTX 3060, sm_86) — skip RTX 5080 (sm_120, unsupported by this torch build)
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import numpy as np
import matplotlib.pyplot as plt
import cv2
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ── Resolve project paths ────────────────────────────────────────────────────
_cwd = Path(os.path.abspath(''))
PROJ_ROOT = _cwd
for _p in [_cwd, _cwd.parent, _cwd.parent.parent, _cwd.parent.parent.parent]:
    if (_p / "pipeline").is_dir() and (_p / "configs").is_dir():
        PROJ_ROOT = _p
        break

DATA_ROOT = PROJ_ROOT.parent / "data"
if not DATA_ROOT.is_dir():
    DATA_ROOT = PROJ_ROOT / "data"

sys.path.insert(0, str(PROJ_ROOT))

print(f"Project root: {PROJ_ROOT}")
print(f"Data root:    {DATA_ROOT}  (exists={DATA_ROOT.is_dir()})")

# ── Config ───────────────────────────────────────────────────────────────────
TEST_TYPE   = "fitness"                                        # fitness | agility | sprint | explosiveness | balance
VIDEO_PATH  = DATA_ROOT / "raw_footage" / "shuttles_test_behind.mp4"
ENABLE_POSE = True
ENABLE_OCR  = False    # set True if paddleocr 2.x is installed

assert VIDEO_PATH.exists(), f"Video not found: {VIDEO_PATH}\n  Check DATA_ROOT above — files are at: {list((DATA_ROOT/'raw_footage').glob('*.mp4')) if (DATA_ROOT/'raw_footage').is_dir() else 'raw_footage/ missing'}"

config_file = PROJ_ROOT / "configs" / "test_configs" / f"{TEST_TYPE}.json"
with open(config_file) as f:
    geometry_config = json.load(f)

TARGET_FPS = geometry_config.get("capture_fps", 15)

import torch
if torch.cuda.is_available():

    
    print(f"GPU:  {torch.cuda.get_device_name(0)}  (CUDA {torch.version.cuda})")
else:
    print("WARNING: CUDA not available — running on CPU")

print(f"\nTest: {TEST_TYPE}  Video: {VIDEO_PATH.name}  FPS: {TARGET_FPS}")

Project root: /home/alex/PycharmProjects/vigour-poc/pipeline-poc
Data root:    /home/alex/PycharmProjects/vigour-poc/data  (exists=True)
GPU:  NVIDIA GeForce RTX 3060  (CUDA 12.1)

Test: fitness  Video: shuttles_test_behind.mp4  FPS: 50


In [ ]:
from pipeline.ingest import extract_frames
from pipeline.detect import PersonDetector
from pipeline.track import PersonTracker
from pipeline.calibrate import Calibrator
from pipeline.models import CalibrationResult

t0 = time.time()

# ── Ingest ───────────────────────────────────────────────────────────────────
print("Stage 1/7: Ingest...", end=" ", flush=True)
raw_frames, frame_indices, timestamps_s = [], [], []
for fi, frame, ts in extract_frames(str(VIDEO_PATH), target_fps=TARGET_FPS):
    raw_frames.append(frame)
    frame_indices.append(fi)
    timestamps_s.append(ts)
print(f"{len(raw_frames)} frames ({time.time()-t0:.1f}s)")

# ── Detect ───────────────────────────────────────────────────────────────────
print("Stage 2/7: Detect...", end=" ", flush=True)
detector = PersonDetector()
all_detections = []
for i, frame in enumerate(raw_frames):
    dets = detector.detect(frame)
    for d in dets:
        d.frame_idx = frame_indices[i]
    all_detections.append(dets)
print(f"done ({time.time()-t0:.1f}s)")

# ── Track ────────────────────────────────────────────────────────────────────
print("Stage 3/7: Track...", end=" ", flush=True)
tracker = PersonTracker()
all_tracks = []
for i, (frame, dets) in enumerate(zip(raw_frames, all_detections)):
    all_tracks.append(tracker.update(dets, frame_idx=frame_indices[i]))
n_ids = len({t.track_id for ft in all_tracks for t in ft})
print(f"{n_ids} track IDs ({time.time()-t0:.1f}s)")

# ── Pose (optional) ─────────────────────────────────────────────────────────
if ENABLE_POSE:
    print("Stage 4/7: Pose...", end=" ", flush=True)
    from pipeline.pose import PoseEstimator
    estimator = PoseEstimator()
    all_poses = []
    for frame, tracks in zip(raw_frames, all_tracks):
        all_poses.append(estimator.estimate_batch(frame, tracks))
    print(f"done ({time.time()-t0:.1f}s)")
else:
    print("Stage 4/7: Pose SKIPPED")
    all_poses = [[] for _ in raw_frames]

# ── OCR (optional) ──────────────────────────────────────────────────────────
resolved = {}
if ENABLE_OCR:
    print("Stage 5/7: OCR...", end=" ", flush=True)
    from pipeline.ocr import BibOCR, resolve_bibs
    ocr = BibOCR()
    frame_readings = []
    for i, (frame, tracks) in enumerate(zip(raw_frames, all_tracks)):
        if i % 5 == 0:
            frame_readings.append(ocr.read_frame(frame, tracks))
    resolved = resolve_bibs(frame_readings)
    print(f"{len(resolved)} bibs ({time.time()-t0:.1f}s)")
else:
    print("Stage 5/7: OCR SKIPPED")

for frame_tracks in all_tracks:
    for track in frame_tracks:
        bib, conf = resolved.get(track.track_id, (None, 0.0))
        track.bib_number = bib
        track.bib_confidence = conf

# ── Calibrate ────────────────────────────────────────────────────────────────
print("Stage 6/7: Calibrate (SAM3)...", end=" ", flush=True)
calibrator = Calibrator(
    detector_backend=geometry_config.get("calibration_detector", "sam3_prompt"),
    sam_model_path=geometry_config.get("calibration_model", "sam3.pt"),
    sam_prompt=geometry_config.get("calibration_prompt", "training cone"),
    min_confidence=geometry_config.get("calibration_min_confidence", 0.5),
)
cone_layout = dict(geometry_config.get("cone_layout", {}))
if "cone_count" not in cone_layout and "cone_count" in geometry_config:
    cone_layout["cone_count"] = geometry_config["cone_count"]

if cone_layout and cone_layout.get("pattern"):
    calibration = calibrator.calibrate_from_layout(raw_frames[0], cone_layout)
else:
    calibration = calibrator.calibrate_single_axis(raw_frames[0])
print(f"{calibration.method} valid={calibration.is_valid} err={calibration.reprojection_error_cm:.1f}cm "
      f"cones={len(calibration.cone_positions_px)} ({time.time()-t0:.1f}s)")

# ── Extract ──────────────────────────────────────────────────────────────────
print("Stage 7/7: Extract metrics...", end=" ", flush=True)
from worker.celery_app import _get_extractor
try:
    extractor = _get_extractor(TEST_TYPE, geometry_config, calibration)
    results = extractor.extract(all_tracks, all_poses, raw_frames)
except Exception as exc:
    print(f"FAILED: {exc}")
    results = []
print(f"{len(results)} results ({time.time()-t0:.1f}s)")

print(f"\nPipeline complete in {time.time()-t0:.0f}s")

Stage 1/7: Ingest... 1775 frames (0.8s)
Stage 2/7: Detect... done (15.5s)
Stage 3/7: Track... 33 track IDs (18.3s)
Stage 4/7: Pose... Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-m_simcc-aic-coco_pt-aic-coco_420e-256x192-63eb25f7_20230126.pth


## 2. Metric Results

In [ ]:
import pandas as pd

# ── Results table ────────────────────────────────────────────────────────────
if results:
    rows = []
    for r in results:
        bib = r.student_bib if r.student_bib and r.student_bib != -1 else None
        label = f"Bib {bib}" if bib else f"Track {r.track_id}"
        row = {
            "Student": label,
            "Track ID": r.track_id,
            f"Result ({r.metric_unit})": round(r.metric_value, 2),
            "Confidence": round(r.confidence_score, 2),
            "Flags": ", ".join(r.flags) if r.flags else "",
        }
        if r.raw_data and "set_distances_m" in r.raw_data:
            for si, sd in enumerate(r.raw_data["set_distances_m"], 1):
                row[f"Set {si} (m)"] = round(sd, 2)
        rows.append(row)

    df = pd.DataFrame(rows).sort_values(f"Result ({results[0].metric_unit})", ascending=False)
    display(HTML(f"<h3>{TEST_TYPE.title()} Test — {len(results)} students</h3>"))
    display(HTML(df.to_html(index=False)))

    # ── Bar chart ────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(max(8, len(results)*0.5), 4))
    metric_col = f"Result ({results[0].metric_unit})"
    bars = ax.barh(df["Student"], df[metric_col], color="steelblue", edgecolor="white")
    ax.set_xlabel(f"{results[0].metric_unit}")
    ax.set_title(f"{TEST_TYPE.title()} — Metric per Student")
    ax.invert_yaxis()
    for bar, val in zip(bars, df[metric_col]):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                f"{val:.1f}", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No results extracted.")

## 3. Calibration Overview

In [ ]:
from pipeline.visualise import render_top_down_view, PipelineVisualiser, VisOptions

# ── Calibration info ─────────────────────────────────────────────────────────
print(f"Method:            {calibration.method}")
print(f"Valid:             {calibration.is_valid}")
print(f"Reprojection err:  {calibration.reprojection_error_cm:.2f} cm")
print(f"Cones detected:    {len(calibration.cone_positions_px)}")
if calibration.condition_number:
    print(f"Condition number:  {calibration.condition_number:.1f}")

# ── First frame: annotated + top-down side by side ──────────────────────────
mid = len(raw_frames) // 2
frame, tracks, poses = raw_frames[mid], all_tracks[mid], all_poses[mid]

# Large standalone top-down
top_down = render_top_down_view(calibration, tracks, poses, size=(500, 500))

# Annotated frame with all overlays
opts = VisOptions(show_top_down_view=False, show_calibration_grid=calibration.is_valid, show_skeleton=ENABLE_POSE)
annotated = PipelineVisualiser.annotate_frame(
    frame, tracks, poses, calibration, results, test_type=TEST_TYPE, opts=opts,
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].imshow(annotated[:, :, ::-1])
axes[0].set_title(f"Annotated Frame {mid}")
axes[0].axis("off")
axes[1].imshow(top_down[:, :, ::-1])
axes[1].set_title("Top-down World View (cm)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 4. Frame Scrubber — Both Views

Drag the slider to scrub through frames. Shows annotated camera view (left) + top-down world view (right) side by side.

In [ ]:
%matplotlib inline

from IPython.display import display, clear_output
import ipywidgets as widgets

vis_opts = VisOptions(
    show_top_down_view=False,
    show_calibration_grid=calibration.is_valid,
    show_skeleton=ENABLE_POSE,
    show_hud=True,
)

output_area = widgets.Output()

def show_frame(idx):
    frame = raw_frames[idx]
    tracks = all_tracks[idx]
    poses = all_poses[idx]
    ts = timestamps_s[idx]

    annotated = PipelineVisualiser.annotate_frame(
        frame, tracks, poses, calibration, results,
        test_type=TEST_TYPE, frame_idx=idx, timestamp_s=ts, opts=vis_opts,
    )
    top_down = render_top_down_view(calibration, tracks, poses, size=(500, 500))

    with output_area:
        clear_output(wait=True)
        fig, (ax_cam, ax_td) = plt.subplots(1, 2, figsize=(16, 5))
        ax_cam.imshow(annotated[:, :, ::-1])
        ax_cam.set_title("Camera View")
        ax_cam.axis("off")
        ax_td.imshow(top_down[:, :, ::-1])
        ax_td.set_title("Top-down (cm)")
        ax_td.axis("off")
        fig.suptitle(f"Frame {idx} / {len(raw_frames)-1}  |  t={ts:.2f}s", fontsize=11)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

slider = widgets.IntSlider(
    value=0, min=0, max=len(raw_frames)-1, step=1,
    description="Frame:", layout=widgets.Layout(width="80%"),
    continuous_update=False,
)

play = widgets.Play(
    value=0, min=0, max=len(raw_frames)-1,
    step=max(1, len(raw_frames) // 100),
    interval=100, description="Play",
)
widgets.jslink((play, "value"), (slider, "value"))

slider.observe(lambda change: show_frame(change["new"]), names="value")

display(widgets.HBox([play, slider]))
display(output_area)
show_frame(0)

## 5. Static Grid — Key Frames

Quick visual overview: 8 evenly-spaced frames with both views.

In [ ]:
%matplotlib inline

N_SAMPLES = 8
indices = np.linspace(0, len(raw_frames)-1, N_SAMPLES, dtype=int)

fig, axes = plt.subplots(2, N_SAMPLES, figsize=(N_SAMPLES*3, 6))

for col, idx in enumerate(indices):
    frame = raw_frames[idx]
    tracks = all_tracks[idx]
    poses = all_poses[idx]
    ts = timestamps_s[idx]

    annotated = PipelineVisualiser.annotate_frame(
        frame, tracks, poses, calibration, results,
        test_type=TEST_TYPE, frame_idx=idx, timestamp_s=ts, opts=vis_opts,
    )
    top_down = render_top_down_view(calibration, tracks, poses, size=(300, 300))

    axes[0, col].imshow(annotated[:, :, ::-1])
    axes[0, col].set_title(f"t={ts:.1f}s", fontsize=8)
    axes[0, col].axis("off")

    axes[1, col].imshow(top_down[:, :, ::-1])
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Camera", fontsize=10)
axes[1, 0].set_ylabel("Top-down", fontsize=10)
fig.suptitle(f"{TEST_TYPE.title()} — {VIDEO_PATH.name}", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Export Annotated Video (with top-down inset)

In [ ]:
output_dir = DATA_ROOT / "annotated"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"dashboard_{TEST_TYPE}_topdown.mp4"

export_opts = VisOptions(
    show_calibration_grid=calibration.is_valid,
    show_skeleton=ENABLE_POSE,
    show_top_down_view=True,
    top_down_view_size=(280, 280),
    show_hud=True,
)

with PipelineVisualiser(output_path, test_type=TEST_TYPE, fps=TARGET_FPS, options=export_opts) as vis:
    for frame, tracks, poses, ts in zip(raw_frames, all_tracks, all_poses, timestamps_s):
        vis.write_frame(frame, tracks, poses, calibration, results, timestamp_s=ts)

print(f"Saved: {output_path}  ({output_path.stat().st_size / 1e6:.1f} MB)")